In [4]:
# ============================================================
# Spectral learning under noise: multi-basis degradation study
#
# Computes and plots three degradation metrics as a function of
# noise amplitude for sparse spectral regression across seven
# orthonormal bases (Fourier, Bessel, Legendre, Jacobi,
# Chebyshev I, Chebyshev II, Haar) in one and two dimensions:
#
#   - Cosine overlap q between noisy and noiseless whitened
#     coefficient vectors
#   - Normalized Euclidean coefficient distance
#     ||c(sigma) - c0||_2 / ||c0||_2
#   - Test root-mean-square error (RMSE)
#
# Results are shown against both absolute noise sigma and
# rescaled noise sigma/sigma*, where sigma* is the intrinsic
# noise scale derived in the paper. When plotted against
# sigma/sigma*, curves from all bases and dimensions collapse
# onto the universal theoretical prediction
# q(sigma) = (1 + (sigma/sigma*)^2)^{-1/2}.
#
# Output: 6 PDF figures saved to the specified output directory.
#
# Dependencies: Random, LinearAlgebra, Statistics, Printf,
#               PyPlot, LaTeXStrings, ProgressMeter, Lasso,
#               Distributions, SpecialFunctions
# ============================================================

using Random, LinearAlgebra, Statistics, Printf
using PyPlot
using LaTeXStrings
using ProgressMeter
using Lasso
using Distributions
using SpecialFunctions

# ------------------------------------------------------------
# Global style variables
# ------------------------------------------------------------
const FONT_TYPE        = "Arial"
const FONT_SIZE        = 18
const LABEL_SIZE       = 18
const TITLE_SIZE       = 16
const TICK_LABEL_SIZE  = 16
const LEGEND_SIZE      = 8

const AXES_LINEWIDTH   = 1.8
const TICK_MAJOR_WIDTH = 1.1
const TICK_MINOR_WIDTH = 0.8
const TICK_MAJOR_SIZE  = 4.5
const TICK_MINOR_SIZE  = 2.5

# ------------------------------------------------------------
# Global caches
# Keys: (mode, d, basis) where mode ∈ (:rel, :abs), d ∈ {1,2}
# ------------------------------------------------------------
global MULTI_BASIS_CACHE = Dict{Tuple{Symbol,Int,Symbol}, NamedTuple}()

# ------------------------------------------------------------
# Small helpers
# ------------------------------------------------------------
ensure_dir(path::String) = (isdir(path) || mkpath(path); path)
interval_length(a, b) = b - a
to_m11(x, a, b) = (2.0 .* (x .- a) ./ (b - a)) .- 1.0
to_01(x, a, b)  = (x .- a) ./ (b - a)

function set_font!(txt; size::Real=FONT_SIZE)
    txt.set_fontname(FONT_TYPE)
    txt.set_fontsize(size)
end

function style_tick_fonts!(ax)
    for lab in ax.get_xticklabels()
        set_font!(lab; size=TICK_LABEL_SIZE)
    end
    for lab in ax.get_yticklabels()
        set_font!(lab; size=TICK_LABEL_SIZE)
    end
end

function style_2d_axes!(ax; xlabel=nothing, ylabel=nothing, title=nothing, labelpad::Real=5)
    ax.tick_params(axis="both", which="major", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MAJOR_WIDTH, length=TICK_MAJOR_SIZE, pad=4)
    ax.tick_params(axis="both", which="minor", labelsize=TICK_LABEL_SIZE,
                   width=TICK_MINOR_WIDTH, length=TICK_MINOR_SIZE, pad=4)

    for side in ("left","bottom","right","top")
        ax.spines[side].set_linewidth(AXES_LINEWIDTH)
    end

    if xlabel !== nothing
        ax.set_xlabel(xlabel, labelpad=labelpad)
        set_font!(ax.xaxis.label; size=LABEL_SIZE)
    end
    if ylabel !== nothing
        ax.set_ylabel(ylabel, labelpad=labelpad)
        set_font!(ax.yaxis.label; size=LABEL_SIZE)
    end
    if title !== nothing
        ax.set_title(title, pad=6)
        set_font!(ax.title; size=TITLE_SIZE)
    end

    style_tick_fonts!(ax)
end

function style_legend!(leg)
    leg === nothing && return
    if hasproperty(leg, :get_title)
        ttl = leg.get_title()
        ttl !== nothing && set_font!(ttl; size=LEGEND_SIZE)
    end
    for txt in leg.get_texts()
        set_font!(txt; size=LEGEND_SIZE)
    end
end

function set_log_ylim_reasonable!(ax, yvals::Vector{Float64}; floor::Float64=1e-12)
    vals = filter(>(floor), yvals)
    if isempty(vals)
        ax.set_ylim(floor, 1.0)
        return
    end
    ymin = minimum(vals)
    ymax = maximum(vals)
    ax.set_ylim(max(0.5*ymin, floor), 1.5*ymax)
end

function basis_display_name(b::Symbol)
    if b == :fourier
        return "Fourier"
    elseif b == :bessel
        return "Bessel"
    elseif b == :legendre
        return "Legendre"
    elseif b == :jacobi
        return "Jacobi"
    elseif b == :chebyshev1
        return "Chebyshev I"
    elseif b == :chebyshev2
        return "Chebyshev II"
    elseif b == :haar
        return "Haar"
    elseif b == :mathieu
        return "Mathieu"
    elseif b == :pswf
        return "PSWF"
    else
        return uppercasefirst(String(b))
    end
end

function split_dim_legend!(ax, handles_1d, labels_1d, handles_2d, labels_2d;
                           loc1="lower left", loc2="lower left",
                           anchor1=nothing, anchor2=nothing)
    kwargs1 = Dict{Symbol,Any}(
        :frameon => false,
        :title => "1D",
        :borderpad => 0.10,
        :labelspacing => 0.18,
        :handletextpad => 0.22,
        :columnspacing => 0.20,
        :markerscale => 0.85,
        :borderaxespad => 0.10,
    )
    kwargs2 = Dict{Symbol,Any}(
        :frameon => false,
        :title => "2D",
        :borderpad => 0.10,
        :labelspacing => 0.18,
        :handletextpad => 0.22,
        :columnspacing => 0.20,
        :markerscale => 0.85,
        :borderaxespad => 0.10,
    )

    if anchor1 !== nothing
        kwargs1[:bbox_to_anchor] = anchor1
    end
    if anchor2 !== nothing
        kwargs2[:bbox_to_anchor] = anchor2
    end

    leg1 = ax.legend(handles_1d, labels_1d, loc=loc1; kwargs1...)
    style_legend!(leg1)
    ax.add_artist(leg1)

    leg2 = ax.legend(handles_2d, labels_2d, loc=loc2; kwargs2...)
    style_legend!(leg2)

    return leg1, leg2
end

function total_degree_multiindex(d::Int, D::Int)
    idx = NTuple{d,Int}[]
    cur = zeros(Int, d)

    function rec!(pos::Int, remaining::Int)
        if pos == d
            cur[pos] = remaining
            push!(idx, Tuple(cur))
            return
        end
        for k in 0:remaining
            cur[pos] = k
            rec!(pos + 1, remaining - k)
        end
    end

    for s in 0:D
        rec!(1, s)
    end
    return idx
end

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
Base.@kwdef mutable struct Config
    seed::Int = 0
    d::Int = 1
    a::Vector{Float64} = [-pi]
    b::Vector{Float64} = [ pi]

    n_train::Int = 300
    n_rep::Int   = 60

    sigma_ratio_min::Float64 = 1e-3
    sigma_ratio_max::Float64 = 20.0
    n_sigma::Int = 80
    include_sigma0::Bool = true

    sigma_abs_min::Float64 = 5e-4
    sigma_abs_max::Float64 = 5e1
    n_sigma_abs::Int = 80
    include_sigma0_abs::Bool = false

    basis::Symbol = :fourier
    K::Int = 12
    D::Int = 18

    nu::Float64 = 0.0
    Nbessel::Int = 30

    mathieu_q::Float64 = 5.0
    Kmathieu::Int = 40

    pswf_c::Float64 = 8.0
    pswf_nquad::Int = 160

    jacobi_alpha::Float64 = -0.25
    jacobi_beta::Float64  = -0.25

    lambda::Float64 = 6e-4
    lasso_maxit::Int = 60_000
    ridge::Float64 = 1e-8
    support_eps::Float64 = 1e-8

    outdir::String = "."
end

# ------------------------------------------------------------
# σ-grid and theory
# ------------------------------------------------------------
function make_sigmas_log(σmin::Float64, σmax::Float64, n_sigma::Int; include_sigma0::Bool=true)
    npos = include_sigma0 ? (n_sigma - 1) : n_sigma
    @assert npos >= 4
    sig_pos = exp.(range(log(σmin), log(σmax); length=npos))
    return include_sigma0 ? vcat(0.0, sig_pos) : sig_pos
end

q_theory_from_x(x::AbstractVector{<:Real}) = (1 .+ (Float64.(x)).^2).^(-0.5)

# ------------------------------------------------------------
# Whitening: centered design first, then whiten
# ------------------------------------------------------------
function invsqrt_gram_centered(Phi::Matrix{Float64}; ridge::Float64)
    N, p = size(Phi)
    μΦ = vec(mean(Phi; dims=1))
    Phi_c = Phi .- reshape(μΦ, 1, :)
    Gc = (Phi_c' * Phi_c) / N
    @inbounds for i in 1:p
        Gc[i,i] += ridge
    end
    F = eigen(Symmetric(Gc))
    λ = F.values
    U = F.vectors
    invsqrtλ = 1.0 ./ sqrt.(max.(λ, 1e-18))
    A = U * Diagonal(invsqrtλ) * U'
    return (A=A, μΦ=μΦ, Phi_c=Phi_c)
end

# ------------------------------------------------------------
# Robust Lasso on PRE-CENTERED design
# ------------------------------------------------------------
function fit_lasso_precentered_robust(Phi::Matrix{Float64}, y::Vector{Float64};
                                      lambda::Float64,
                                      maxit::Int,
                                      cd_tol::Float64=5e-5,
                                      max_retries::Int=3)
    μy = mean(y)
    y_c = y .- μy
    λtry = lambda

    for _ in 1:max_retries
        try
            path = fit(LassoPath, Phi, y_c, Normal();
                       α=1.0, λ=[λtry],
                       standardize=false, intercept=false,
                       cd_maxiter=maxit,
                       cd_tol=cd_tol)

            B = Lasso.coef(path; select=Lasso.AllSeg())
            c = Vector(B[:,1])
            return (μy=μy, c=c, lambda_used=λtry)
        catch
            λtry *= 2.0
        end
    end

    G = Phi' * Phi
    p = size(G,1)
    G .+= (lambda * 10.0) * I(p)
    c = G \ (Phi' * y_c)
    return (μy=μy, c=c, lambda_used=:ridge_fallback)
end

# ------------------------------------------------------------
# Quadrature helpers
# ------------------------------------------------------------
function gausslegendre(n::Int)
    @assert n >= 2
    k = 1:(n-1)
    β = k ./ sqrt.(4 .* k.^2 .- 1.0)
    T = SymTridiagonal(zeros(Float64, n), Float64.(β))
    F = eigen(T)
    x = F.values
    V = F.vectors
    w = 2.0 .* (V[1, :].^2)
    return x, w
end

function orthonormalize_family_from_quadrature(Bx_raw::Matrix{Float64},
                                               Bq_raw::Matrix{Float64},
                                               wq::Vector{Float64})
    G = Bq_raw' * (Bq_raw .* reshape(wq, :, 1))
    F = eigen(Symmetric(G))
    λ = max.(F.values, 1e-18)
    U = F.vectors
    T = U * Diagonal(1.0 ./ sqrt.(λ)) * U'
    return Bx_raw * T
end

function interval_orthonormalized_family(x::Vector{Float64}, D::Int, a::Float64, b::Float64,
                                         rawfun::Function; nquad::Int=max(80, 4D + 20))
    z = to_m11(x, a, b)
    Bx_raw = rawfun(z, D)

    zq, wq = gausslegendre(nquad)
    Bq_raw = rawfun(zq, D)

    L = interval_length(a, b)
    wx = (L / 2.0) .* wq

    return orthonormalize_family_from_quadrature(Bx_raw, Bq_raw, wx)
end

# ------------------------------------------------------------
# Basis coordinate maps
# ------------------------------------------------------------
function to_theta_pm_pi(X::Matrix{Float64}, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Θ = zeros(Float64, N, d)
    for j in 1:d
        Θ[:, j] .= (2π .* (X[:, j] .- a[j]) ./ (b[j] - a[j])) .- π
    end
    return Θ
end

function to_theta_0_2pi(X::Matrix{Float64}, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Θ = zeros(Float64, N, d)
    for j in 1:d
        Θ[:, j] .= 2π .* (X[:, j] .- a[j]) ./ (b[j] - a[j])
    end
    return Θ
end

# ------------------------------------------------------------
# Fourier
# ------------------------------------------------------------
function fourier_1d_block(theta::Vector{Float64}, K::Int, L::Float64)
    N = length(theta)
    B = zeros(Float64, N, 2K + 1)
    B[:, 1] .= 1.0 / sqrt(L)
    sfac = sqrt(2.0 / L)
    for k in 1:K
        B[:, 1 + k]     .= sfac .* cos.(k .* theta)
        B[:, 1 + K + k] .= sfac .* sin.(k .* theta)
    end
    return B
end

@inline function fourier_1d_col(kind::Symbol, k::Int, K::Int)
    if kind === :const
        return 1
    elseif kind === :cos
        return 1 + k
    elseif kind === :sin
        return 1 + K + k
    else
        error("Unknown Fourier 1D kind=$(kind).")
    end
end

function build_fourier_design(X::Matrix{Float64}, K::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Θ = to_theta_pm_pi(X, a, b)

    if d == 1
        L = interval_length(a[1], b[1])
        B = fourier_1d_block(vec(Θ[:, 1]), K, L)
        return B[:, 2:end], nothing
    end

    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        Lj = interval_length(a[j], b[j])
        Φdim[j] = fourier_1d_block(vec(Θ[:, j]), K, Lj)
    end

    idx = total_degree_multiindex(d, K)
    cols = Vector{Vector{Float64}}()

    for α in idx
        all(==(0), α) && continue
        active = findall(>(0), α)
        nactive = length(active)

        for trig_choice in Iterators.product(ntuple(_ -> (:cos, :sin), nactive)...)
            col = ones(Float64, N)
            @inbounds for (m, j) in enumerate(active)
                k = α[j]
                col .*= @view(Φdim[j][:, fourier_1d_col(trig_choice[m], k, K)])
            end
            push!(cols, col)
        end
    end

    Φ = zeros(Float64, N, length(cols))
    for i in 1:length(cols)
        Φ[:, i] .= cols[i]
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# Legendre
# ------------------------------------------------------------
function legendre_polys(z::Vector{Float64}, D::Int)
    N = length(z)
    P = zeros(Float64, N, D+1)
    P[:, 1] .= 1.0
    if D ≥ 1
        P[:, 2] .= z
    end
    for n in 1:(D-1)
        P[:, n+2] .= ((2n+1) .* z .* P[:, n+1] .- n .* P[:, n]) ./ (n+1)
    end
    return P
end

function legendre_orthonormal_interval(x::Vector{Float64}, D::Int, a::Float64, b::Float64)
    z = to_m11(x, a, b)
    P = legendre_polys(z, D)
    L = interval_length(a, b)
    for n in 0:D
        P[:, n+1] .*= sqrt((2n+1) / L)
    end
    return P
end

function build_legendre_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        Φdim[j] = legendre_orthonormal_interval(vec(X[:, j]), D, a[j], b[j])
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))
    for (k, α) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, α[j] + 1])
        end
        Φ[:, k] = col
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# Jacobi / Chebyshev helpers
# ------------------------------------------------------------
function jacobi_polys(z::Vector{Float64}, D::Int, α::Float64, β::Float64)
    N = length(z)
    P = zeros(Float64, N, D+1)
    P[:,1] .= 1.0

    if D >= 1
        P[:,2] .= 0.5 .* ((2.0 + α + β) .* z .+ (α - β))
    end

    for n in 1:(D-1)
        nf = Float64(n)
        a1 = 2.0 * (nf + 1.0) * (nf + α + β + 1.0) * (2.0*nf + α + β)
        a2 = (2.0*nf + α + β + 1.0) .* (
                (2.0*nf + α + β + 2.0) .* (2.0*nf + α + β) .* z .+
                (α^2 - β^2)
             )
        a3 = 2.0 * (nf + α) * (nf + β) * (2.0*nf + α + β + 2.0)
        P[:, n+2] .= (a2 .* P[:, n+1] .- a3 .* P[:, n]) ./ a1
    end

    return P
end

function safe_endpoint_weight(z::Vector{Float64}, pleft::Float64, pright::Float64; eps::Float64=1e-14)
    zl = clamp.(1.0 .- z, eps, Inf)
    zr = clamp.(1.0 .+ z, eps, Inf)
    return zl.^pleft .* zr.^pright
end

# ------------------------------------------------------------
# Jacobi
# ------------------------------------------------------------
function jacobi_functions_general(z::Vector{Float64}, D::Int, α::Float64, β::Float64)
    P = jacobi_polys(z, D, α, β)
    return P .* safe_endpoint_weight(z, α/2, β/2)
end

function jacobi_orthonormal_interval(x::Vector{Float64}, D::Int, a::Float64, b::Float64,
                                     α::Float64, β::Float64)
    return interval_orthonormalized_family(x, D, a, b,
        (z, Dloc) -> jacobi_functions_general(z, Dloc, α, β))
end

function build_jacobi_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64},
                             α::Float64, β::Float64)
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)

    for j in 1:d
        Φdim[j] = jacobi_orthonormal_interval(vec(X[:, j]), D, a[j], b[j], α, β)
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))

    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end

    return Φ, nothing
end

# ------------------------------------------------------------
# Chebyshev I
# ------------------------------------------------------------
function chebyshevT_polys(z::Vector{Float64}, D::Int)
    N = length(z)
    T = zeros(Float64, N, D+1)
    T[:,1] .= 1.0
    if D >= 1
        T[:,2] .= z
    end
    for n in 1:(D-1)
        T[:,n+2] .= 2.0 .* z .* T[:,n+1] .- T[:,n]
    end
    return T
end

function chebyshev1_functions(z::Vector{Float64}, D::Int)
    T = chebyshevT_polys(z, D)
    w12 = clamp.(1.0 .- z.^2, 1e-14, Inf).^(-0.25)
    return T .* w12
end

function chebyshev1_orthonormal_interval(x::Vector{Float64}, D::Int, a::Float64, b::Float64)
    return interval_orthonormalized_family(x, D, a, b, chebyshev1_functions)
end

function build_chebyshev1_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)

    for j in 1:d
        Φdim[j] = chebyshev1_orthonormal_interval(vec(X[:, j]), D, a[j], b[j])
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))

    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end

    return Φ, nothing
end

# ------------------------------------------------------------
# Chebyshev II
# ------------------------------------------------------------
function chebyshevU_polys(z::Vector{Float64}, D::Int)
    N = length(z)
    U = zeros(Float64, N, D+1)
    U[:,1] .= 1.0
    if D >= 1
        U[:,2] .= 2.0 .* z
    end
    for n in 1:(D-1)
        U[:,n+2] .= 2.0 .* z .* U[:,n+1] .- U[:,n]
    end
    return U
end

function chebyshev2_functions(z::Vector{Float64}, D::Int)
    U = chebyshevU_polys(z, D)
    w12 = clamp.(1.0 .- z.^2, 1e-14, Inf).^(0.25)
    return U .* w12
end

function chebyshev2_orthonormal_interval(x::Vector{Float64}, D::Int, a::Float64, b::Float64)
    return interval_orthonormalized_family(x, D, a, b, chebyshev2_functions)
end

function build_chebyshev2_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)

    for j in 1:d
        Φdim[j] = chebyshev2_orthonormal_interval(vec(X[:, j]), D, a[j], b[j])
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))

    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end

    return Φ, nothing
end

# ------------------------------------------------------------
# Bessel
# ------------------------------------------------------------
function besselj_zero(ν::Float64, n::Int; tol::Float64=1e-12, maxit::Int=200)
    @assert n >= 1
    x = (n + 0.5*ν - 0.25) * π
    step = π/2
    L = max(x - step, 1e-12)
    R = x + step

    fL = besselj(ν, L)
    fR = besselj(ν, R)

    it = 0
    while fL*fR > 0 && it < 2000
        step *= 1.2
        L = max(x - step, 1e-12)
        R = x + step
        fL = besselj(ν, L)
        fR = besselj(ν, R)
        it += 1
    end
    if fL*fR > 0
        error("Failed to bracket root for J_ν with ν=$ν, n=$n")
    end

    for _ in 1:maxit
        M = 0.5*(L+R)
        fM = besselj(ν, M)
        if abs(fM) < tol || (R-L) < tol
            return M
        end
        if fL*fM <= 0
            R = M
            fR = fM
        else
            L = M
            fL = fM
        end
    end
    return 0.5*(L+R)
end

function bessel_zero_list(ν::Float64, N::Int)
    α = zeros(Float64, N)
    for n in 1:N
        α[n] = besselj_zero(ν, n)
    end
    return α
end

function bessel_basis_orthonormal_unitinterval(r::Vector{Float64}, ν::Float64, α::Vector{Float64})
    N = length(r)
    M = length(α)
    Φ = zeros(Float64, N, M)
    sr = sqrt.(max.(r, 0.0))
    for n in 1:M
        an = α[n]
        normfac = sqrt(2.0) / max(abs(besselj(ν + 1.0, an)), 1e-18)
        Φ[:, n] .= normfac .* sr .* besselj.(ν, an .* r)
    end
    return Φ
end

function build_bessel_design(X::Matrix{Float64}, Nb::Int, ν::Float64, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    α = bessel_zero_list(ν, Nb)

    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        r = clamp.(to_01(vec(X[:, j]), a[j], b[j]), 0.0, 1.0)
        Lj = interval_length(a[j], b[j])
        Φdim[j] = bessel_basis_orthonormal_unitinterval(r, ν, α) ./ sqrt(Lj)
    end

    if d == 1
        return Φdim[1], nothing
    end

    Dm = min(D, Nb - 1)
    idx = total_degree_multiindex(d, Dm)
    Φ = zeros(Float64, N, length(idx))
    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# Haar
# ------------------------------------------------------------
function haar_n_to_jk(n::Int)
    @assert n >= 1
    m = n - 1
    j = 0
    cnt = 1
    while m >= cnt
        m -= cnt
        j += 1
        cnt = 2^j
    end
    return j, m
end

function haar_eval_one(r::Float64, n::Int)
    if n == 0
        return (0.0 <= r < 1.0) ? 1.0 : 0.0
    end
    j, k = haar_n_to_jk(n)
    a = k / 2.0^j
    mid = (k + 0.5) / 2.0^j
    b = (k + 1.0) / 2.0^j
    if !(a <= r < b)
        return 0.0
    end
    amp = 2.0^(j/2)
    return (r < mid) ? amp : -amp
end

function haar_orthonormal_subset_unitinterval(r::Vector{Float64}, D::Int)
    N = length(r)
    Φ = zeros(Float64, N, D+1)
    for i in 1:N
        ri = r[i]
        @inbounds for n in 0:D
            Φ[i, n+1] = haar_eval_one(ri, n)
        end
    end
    return Φ
end

function build_haar_design(X::Matrix{Float64}, D::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        r = clamp.(to_01(vec(X[:, j]), a[j], b[j]), 0.0, prevfloat(1.0))
        Lj = interval_length(a[j], b[j])
        Φdim[j] = haar_orthonormal_subset_unitinterval(r, D) ./ sqrt(Lj)
    end

    if d == 1
        return Φdim[1], nothing
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))
    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# Mathieu
# ------------------------------------------------------------
function fourier_real_orthonormal_matrix_interval(theta::Vector{Float64}, K::Int, L::Float64)
    N = length(theta)
    p = 1 + 2K
    B = zeros(Float64, N, p)
    B[:,1] .= 1.0 / sqrt(L)
    if K >= 1
        sfac = sqrt(2.0 / L)
        for k in 1:K
            B[:, 1 + k]     .= sfac .* cos.(k .* theta)
            B[:, 1 + K + k] .= sfac .* sin.(k .* theta)
        end
    end
    return B
end

function mathieu_operator_matrix(K::Int, q::Float64)
    p = 1 + 2K
    L = zeros(Float64, p, p)

    for k in 1:K
        L[1 + k, 1 + k] += k^2
        L[1 + K + k, 1 + K + k] += k^2
    end

    if K >= 2
        ic2 = 1 + 2
        L[1, ic2] += sqrt(2.0) * q
        L[ic2, 1] += sqrt(2.0) * q

        for k in 1:(K-2)
            i  = 1 + k
            ip = 1 + (k+2)
            L[i, ip] += q
            L[ip, i] += q
        end
    end

    if K >= 3
        for k in 1:(K-2)
            i  = 1 + K + k
            ip = 1 + K + (k+2)
            L[i, ip] += q
            L[ip, i] += q
        end
    end

    return Symmetric(L)
end

function mathieu_modes(K::Int, q::Float64, D::Int)
    L = mathieu_operator_matrix(K, q)
    F = eigen(L)
    nkeep = min(D+1, size(L,1))
    return F.vectors[:, 1:nkeep]
end

function build_mathieu_design(X::Matrix{Float64}, D::Int, K::Int, q::Float64, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    Vsel = mathieu_modes(K, q, D)

    Φdim = Vector{Matrix{Float64}}(undef, d)
    Θ = to_theta_0_2pi(X, a, b)
    for j in 1:d
        θ = vec(Θ[:, j])
        Lj = interval_length(a[j], b[j])
        Bθ = fourier_real_orthonormal_matrix_interval(θ, K, Lj)
        Φdim[j] = Bθ * Vsel
    end

    if d == 1
        return Φdim[1], nothing
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))
    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# PSWF
# ------------------------------------------------------------
@inline function pswf_kernel(c::Float64, x::Float64, t::Float64)
    u = x - t
    abs(u) < 1e-14 ? c / π : sin(c*u) / (π*u)
end

function pswf_nystrom(c::Float64, nquad::Int, D::Int)
    t, w = gausslegendre(nquad)
    sqrtw = sqrt.(w)
    A = zeros(Float64, nquad, nquad)
    @inbounds for i in 1:nquad
        ti = t[i]
        si = sqrtw[i]
        for j in 1:nquad
            A[i,j] = si * pswf_kernel(c, ti, t[j]) * sqrtw[j]
        end
    end
    F = eigen(Symmetric(A))
    vals = F.values
    vecs = F.vectors

    perm = sortperm(vals; rev=true)
    vals = vals[perm]
    vecs = vecs[:, perm]

    nkeep = min(D+1, nquad)
    lam = vals[1:nkeep]
    U = vecs[:, 1:nkeep]
    psi_nodes = U ./ sqrtw

    for n in 1:nkeep
        psi_nodes[:,n] ./= sqrt(max(sum(w .* (psi_nodes[:,n].^2)), 1e-18))
    end

    return t, w, lam, psi_nodes
end

function pswf_eval_from_nodes(c::Float64, x::Vector{Float64}, t::Vector{Float64}, w::Vector{Float64},
                              lam::Vector{Float64}, psi_nodes::Matrix{Float64})
    N = length(x)
    nkeep = length(lam)
    Φ = zeros(Float64, N, nkeep)
    @inbounds for i in 1:N
        xi = x[i]
        for n in 1:nkeep
            s = 0.0
            for j in 1:length(t)
                s += w[j] * pswf_kernel(c, xi, t[j]) * psi_nodes[j, n]
            end
            Φ[i, n] = s / max(lam[n], 1e-18)
        end
    end
    return Φ
end

function build_pswf_design(X::Matrix{Float64}, D::Int, c::Float64, nquad::Int, a::Vector{Float64}, b::Vector{Float64})
    N, d = size(X)
    t, w, lam, psi_nodes = pswf_nystrom(c, nquad, D)

    Φdim = Vector{Matrix{Float64}}(undef, d)
    for j in 1:d
        z = clamp.(to_m11(vec(X[:, j]), a[j], b[j]), -1.0, 1.0)
        Φz = pswf_eval_from_nodes(c, z, t, w, lam, psi_nodes)
        Lj = interval_length(a[j], b[j])
        Φdim[j] = sqrt(2.0 / Lj) .* Φz
    end

    if d == 1
        return Φdim[1], nothing
    end

    idx = total_degree_multiindex(d, D)
    Φ = zeros(Float64, N, length(idx))
    for (k, αidx) in enumerate(idx)
        col = ones(Float64, N)
        @inbounds for j in 1:d
            col .*= @view(Φdim[j][:, αidx[j] + 1])
        end
        Φ[:, k] = col
    end
    return Φ, nothing
end

# ------------------------------------------------------------
# Dispatcher
# ------------------------------------------------------------
function build_design(cfg::Config, X::Matrix{Float64})
    if cfg.basis == :fourier
        return build_fourier_design(X, cfg.K, cfg.a, cfg.b)
    elseif cfg.basis == :bessel
        return build_bessel_design(X, cfg.Nbessel, cfg.nu, cfg.D, cfg.a, cfg.b)
    elseif cfg.basis == :legendre
        return build_legendre_design(X, cfg.D, cfg.a, cfg.b)
    elseif cfg.basis == :jacobi
        return build_jacobi_design(X, cfg.D, cfg.a, cfg.b, cfg.jacobi_alpha, cfg.jacobi_beta)
    elseif cfg.basis == :chebyshev1
        return build_chebyshev1_design(X, cfg.D, cfg.a, cfg.b)
    elseif cfg.basis == :chebyshev2
        return build_chebyshev2_design(X, cfg.D, cfg.a, cfg.b)
    elseif cfg.basis == :haar
        return build_haar_design(X, cfg.D, cfg.a, cfg.b)
    elseif cfg.basis == :mathieu
        return build_mathieu_design(X, cfg.D, cfg.Kmathieu, cfg.mathieu_q, cfg.a, cfg.b)
    elseif cfg.basis == :pswf
        return build_pswf_design(X, cfg.D, cfg.pswf_c, cfg.pswf_nquad, cfg.a, cfg.b)
    else
        error("Unknown basis=$(cfg.basis).")
    end
end

# ------------------------------------------------------------
# Compute sweep
# ------------------------------------------------------------
function compute_q_and_dist(cfg::Config; f_true::Function, mode::Symbol=:rel, use_threads::Bool=true)
    @assert mode == :rel || mode == :abs

    Random.seed!(cfg.seed)
    ensure_dir(cfg.outdir)

    function sample_X(n)
        X = zeros(Float64, n, cfg.d)
        for j in 1:cfg.d
            X[:,j] .= cfg.a[j] .+ rand(n) .* (cfg.b[j] - cfg.a[j])
        end
        return X
    end

    Xtr = sample_X(cfg.n_train)
    ytr_true = Float64[f_true(view(Xtr, i, :)) for i in 1:cfg.n_train]

    Φtr0, _ = build_design(cfg, Xtr)
    prep = invsqrt_gram_centered(Φtr0; ridge=cfg.ridge)
    A   = prep.A
    μΦ  = prep.μΦ
    Φtr = prep.Phi_c * A

    n_test = 2000
    Xte = sample_X(n_test)
    yte_true = Float64[f_true(view(Xte, i, :)) for i in 1:n_test]
    Φte0, _ = build_design(cfg, Xte)
    Φte = (Φte0 .- reshape(μΦ, 1, :)) * A

    fit0 = fit_lasso_precentered_robust(Φtr, ytr_true; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
    c0 = fit0.c
    c0norm = max(norm(c0), 1e-18)

    p_eff = max(count(abs.(c0) .> cfg.support_eps), 1)
    σstar = c0norm * sqrt(cfg.n_train / p_eff)

    if mode == :rel
        σmin = max(cfg.sigma_ratio_min * σstar, 1e-18)
        σmax = max(cfg.sigma_ratio_max * σstar, σmin * 10)
        sigmas = make_sigmas_log(σmin, σmax, cfg.n_sigma; include_sigma0=cfg.include_sigma0)
    else
        sigmas = make_sigmas_log(cfg.sigma_abs_min, cfg.sigma_abs_max, cfg.n_sigma_abs; include_sigma0=cfg.include_sigma0_abs)
    end

    x = (mode == :rel) ? (sigmas ./ max(σstar, 1e-18)) : sigmas
    x_for_theory = sigmas ./ max(σstar, 1e-18)
    qth = q_theory_from_x(x_for_theory)
    dth_scaled = x_for_theory

    R = cfg.n_rep
    nσ = length(sigmas)

    q_mean    = zeros(Float64, nσ)
    d_mean    = zeros(Float64, nσ)
    rmse_mean = zeros(Float64, nσ)
    q_med     = zeros(Float64, nσ)
    d_med     = zeros(Float64, nσ)
    rmse_med  = zeros(Float64, nσ)

    prog = Progress(nσ; desc="σ-sweep ($(mode), $(cfg.d)D, $(basis_display_name(cfg.basis)))", dt=0.15)

    if use_threads && Threads.nthreads() > 1
        Threads.@threads for si in 1:nσ
            σ = sigmas[si]

            qs_vec = zeros(Float64, R)
            ds_vec = zeros(Float64, R)
            es_vec = zeros(Float64, R)

            for r in 1:R
                rng = MersenneTwister(cfg.seed + 50_000 + 10_000*si + r + (mode == :abs ? 1_000_000 : 0))
                ytr = ytr_true .+ σ .* randn(rng, cfg.n_train)

                fit = fit_lasso_precentered_robust(Φtr, ytr; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
                c = fit.c

                qs_vec[r] = dot(c, c0) / (max(norm(c), 1e-18) * c0norm)
                ds_vec[r] = norm(c - c0)
                yhat = fit.μy .+ Φte * c
                es_vec[r] = sqrt(mean((yhat .- yte_true).^2))
            end

            q_mean[si]    = mean(qs_vec)
            d_mean[si]    = mean(ds_vec)
            rmse_mean[si] = mean(es_vec)
            q_med[si]     = median(qs_vec)
            d_med[si]     = median(ds_vec)
            rmse_med[si]  = median(es_vec)
        end
        for _ in 1:nσ
            next!(prog)
        end
    else
        for si in 1:nσ
            σ = sigmas[si]

            qs_vec = zeros(Float64, R)
            ds_vec = zeros(Float64, R)
            es_vec = zeros(Float64, R)

            for r in 1:R
                rng = MersenneTwister(cfg.seed + 50_000 + 10_000*si + r + (mode == :abs ? 1_000_000 : 0))
                ytr = ytr_true .+ σ .* randn(rng, cfg.n_train)

                fit = fit_lasso_precentered_robust(Φtr, ytr; lambda=cfg.lambda, maxit=cfg.lasso_maxit)
                c = fit.c

                qs_vec[r] = dot(c, c0) / (max(norm(c), 1e-18) * c0norm)
                ds_vec[r] = norm(c - c0)
                yhat = fit.μy .+ Φte * c
                es_vec[r] = sqrt(mean((yhat .- yte_true).^2))
            end

            q_mean[si]    = mean(qs_vec)
            d_mean[si]    = mean(ds_vec)
            rmse_mean[si] = mean(es_vec)
            q_med[si]     = median(qs_vec)
            d_med[si]     = median(ds_vec)
            rmse_med[si]  = median(es_vec)

            next!(prog)
        end
    end

    return x, sigmas, q_mean, q_med, d_mean, d_med, rmse_mean, rmse_med,
           qth, dth_scaled, σstar, c0norm, p_eff
end

# ------------------------------------------------------------
# Main driver
# ------------------------------------------------------------
function plot_basis_comparison_all(bases::Vector{Symbol}, markers::Dict{Symbol,String};
                                   outdir::String="out_multi_basis",
                                   use_threads::Bool=true,
                                   use_median::Bool=true)

    ensure_dir(outdir)

    f1(x) = x[1]^2
    f2(x) = sin(x[1]^2 * exp(x[2]))

    cfg1 = Config(
        seed=0, d=1, a=[-pi], b=[pi],
        n_train=300, n_rep=240,
        sigma_ratio_min=5e-4, sigma_ratio_max=5e2, n_sigma=120, include_sigma0=true,
        sigma_abs_min=5e-4, sigma_abs_max=5e2, n_sigma_abs=120, include_sigma0_abs=false,
        K=15, D=30,
        nu=0.0, Nbessel=30,
        mathieu_q=5.0, Kmathieu=40,
        pswf_c=8.0, pswf_nquad=160,
        lambda=6e-4, lasso_maxit=40_000,
        ridge=1e-8, support_eps=1e-4,
        outdir=outdir
    )

    cfg2 = Config(
        seed=1, d=2, a=[-1.0, -1.0], b=[1.0, 1.0],
        n_train=1200, n_rep=120,
        sigma_ratio_min=5e-4, sigma_ratio_max=5e2, n_sigma=120, include_sigma0=true,
        sigma_abs_min=5e-4, sigma_abs_max=5e2, n_sigma_abs=120, include_sigma0_abs=false,
        K=9, D=18,
        nu=0.0, Nbessel=18,
        mathieu_q=5.0, Kmathieu=40,
        pswf_c=8.0, pswf_nquad=160,
        lambda=6e-4, lasso_maxit=60_000,
        ridge=1e-8, support_eps=1e-4,
        outdir=outdir
    )

    empty!(MULTI_BASIS_CACHE)

    for b in bases
        for mode in (:rel, :abs)
            c1 = deepcopy(cfg1)
            c1.basis = b
            x, sigmas, q, qmed, d, dmed, rmse, rmsemed, qth, dth_scaled, σstar, c0norm, p_eff =
                compute_q_and_dist(c1; f_true=f1, mode=mode, use_threads=use_threads)

            MULTI_BASIS_CACHE[(mode,1,b)] = (
                x=x, sigmas=sigmas, q=q, q_med=qmed, d=d, d_med=dmed,
                rmse=rmse, rmse_med=rmsemed,
                qth=qth, dth_scaled=dth_scaled,
                σstar=σstar, c0norm=c0norm, p_eff=p_eff, cfg=c1
            )

            if mode == :rel
                @printf("σ* (1D, %-12s) = %.6g   ||ĉ0||=%.6g   p_eff=%d\n",
                        basis_display_name(b), σstar, c0norm, p_eff)
            end

            c2 = deepcopy(cfg2)
            c2.basis = b
            x2, sigmas2, q2, qmed2, d2, dmed2, rmse2, rmsemed2, qth2, dth_scaled2, σstar2, c0norm2, p_eff2 =
                compute_q_and_dist(c2; f_true=f2, mode=mode, use_threads=use_threads)

            MULTI_BASIS_CACHE[(mode,2,b)] = (
                x=x2, sigmas=sigmas2, q=q2, q_med=qmed2, d=d2, d_med=dmed2,
                rmse=rmse2, rmse_med=rmsemed2,
                qth=qth2, dth_scaled=dth_scaled2,
                σstar=σstar2, c0norm=c0norm2, p_eff=p_eff2, cfg=c2
            )

            if mode == :rel
                @printf("σ* (2D, %-12s) = %.6g   ||ĉ0||=%.6g   p_eff=%d\n",
                        basis_display_name(b), σstar2, c0norm2, p_eff2)
            end
        end
    end

    get_overlap(r) = use_median ? r.q_med : r.q
    get_dist(r)    = use_median ? r.d_med : r.d
    get_rmse(r)    = use_median ? r.rmse_med : r.rmse

    function make_3_plots(mode::Symbol)
        @assert mode == :rel || mode == :abs
        suffix = (mode == :rel) ? "_rel" : "_abs"
        xlab = (mode == :rel) ? L"\sigma/\sigma^{*}" : L"\sigma"

        # =========================
        # FIGURE 1: overlap q
        # =========================
        fig = figure(figsize=(9,5))
        ax = gca()
        ax.set_xscale("log")

        legend_handles_1d = Any[]
        legend_labels_1d  = String[]
        legend_handles_2d = Any[]
        legend_labels_2d  = String[]
        yvals = Float64[]

        if mode == :rel
            b0 = bases[1]
            r0 = MULTI_BASIS_CACHE[(mode,1,b0)]
            h_th, = ax.plot(r0.x, clamp.(r0.qth, 0.0, 1.0), color="black", ls="--", lw=2.5, zorder=1)
            push!(legend_handles_1d, h_th)
            push!(legend_labels_1d, "Theory")
            append!(yvals, clamp.(r0.qth, 0.0, 1.0))

            for b in bases
                r1 = MULTI_BASIS_CACHE[(mode,1,b)]
                r2 = MULTI_BASIS_CACHE[(mode,2,b)]
                ax.plot(r1.x, clamp.(r1.qth, 0.0, 1.0), color="black", ls="--", lw=2.0, alpha=0.25, zorder=1)
                ax.plot(r2.x, clamp.(r2.qth, 0.0, 1.0), color="black", ls="--", lw=2.0, alpha=0.25, zorder=1)
                append!(yvals, clamp.(r1.qth, 0.0, 1.0))
                append!(yvals, clamp.(r2.qth, 0.0, 1.0))
            end
        end

        for b in bases
            nm = basis_display_name(b)

            r1 = MULTI_BASIS_CACHE[(mode,1,b)]
            y1 = clamp.(get_overlap(r1), 0.0, 1.0)
            h1 = ax.scatter(r1.x, y1; marker=markers[b], c="red", s=18, zorder=3)
            push!(legend_handles_1d, h1)
            push!(legend_labels_1d, nm)
            append!(yvals, y1)

            r2 = MULTI_BASIS_CACHE[(mode,2,b)]
            y2 = clamp.(get_overlap(r2), 0.0, 1.0)
            h2 = ax.scatter(r2.x, y2; marker=markers[b], c="blue", s=18, zorder=3)
            push!(legend_handles_2d, h2)
            push!(legend_labels_2d, nm)
            append!(yvals, y2)
        end

        if mode == :rel
            ax.axvline(1.0, lw=2.0, ls=":", color="black", alpha=0.9)
        end

        ax.set_ylim(minimum(yvals), maximum(yvals))
        ax.grid(true, which="both", alpha=0.25)

        style_2d_axes!(ax; xlabel=xlab, ylabel=L"q")
        split_dim_legend!(ax, legend_handles_1d, legend_labels_1d, legend_handles_2d, legend_labels_2d;
                          loc1="lower left", loc2="lower left",
                          anchor1=(0.02, 0.02), anchor2=(0.15, 0.02))

        tight_layout()
        savefig(joinpath(outdir, "basis_overlap$(suffix).pdf"); dpi=300)
        close(fig)

        # =========================
        # FIGURE 2: coefficient distance
        # =========================
        fig = figure(figsize=(9,5))
        ax = gca()
        ax.set_xscale("log")
        ax.set_yscale("log")

        legend_handles_1d = Any[]
        legend_labels_1d  = String[]
        legend_handles_2d = Any[]
        legend_labels_2d  = String[]
        yvals_log = Float64[]

        if mode == :rel
            b0 = bases[1]
            r0 = MULTI_BASIS_CACHE[(mode,1,b0)]
            yth = clamp.(r0.dth_scaled, 1e-18, Inf)
            h_th2, = ax.plot(r0.x, yth, color="k", ls="--", lw=2.2, alpha=0.8, zorder=2)
            push!(legend_handles_1d, h_th2)
            push!(legend_labels_1d, "σ/σ*")
            append!(yvals_log, yth)
        end

        for b in bases
            nm = basis_display_name(b)
            r1 = MULTI_BASIS_CACHE[(mode,1,b)]
            r2 = MULTI_BASIS_CACHE[(mode,2,b)]

            y1 = (mode == :rel) ? (get_dist(r1) ./ max(r1.c0norm, 1e-18)) : get_dist(r1)
            y2 = (mode == :rel) ? (get_dist(r2) ./ max(r2.c0norm, 1e-18)) : get_dist(r2)

            y1p = clamp.(y1, 1e-18, Inf)
            y2p = clamp.(y2, 1e-18, Inf)

            h1 = ax.scatter(r1.x, y1p; marker=markers[b], c="red", s=18, zorder=3)
            h2 = ax.scatter(r2.x, y2p; marker=markers[b], c="blue", s=18, zorder=3)

            push!(legend_handles_1d, h1)
            push!(legend_labels_1d, nm)
            push!(legend_handles_2d, h2)
            push!(legend_labels_2d, nm)

            append!(yvals_log, y1p)
            append!(yvals_log, y2p)
        end

        if mode == :rel
            ax.axvline(1.0, lw=2.0, ls=":", color="black", alpha=0.9)
        end

        set_log_ylim_reasonable!(ax, yvals_log; floor=1e-12)
        ax.grid(true, which="both", alpha=0.25)

        if mode == :rel
            style_2d_axes!(ax; xlabel=xlab, ylabel=L"\|\hat c(\sigma)-\hat c_0\|_2 / \|\hat c_0\|_2")
            split_dim_legend!(ax, legend_handles_1d, legend_labels_1d, legend_handles_2d, legend_labels_2d;
                              loc1="upper left", loc2="upper left",
                              anchor1=(0.02, 0.98), anchor2=(0.15, 0.98))
        else
            style_2d_axes!(ax; xlabel=xlab, ylabel=L"\|\hat c(\sigma)-\hat c_0\|_2")
            split_dim_legend!(ax, legend_handles_1d, legend_labels_1d, legend_handles_2d, legend_labels_2d;
                              loc1="upper left", loc2="upper left",
                              anchor1=(0.02, 0.98), anchor2=(0.15, 0.98))
        end

        tight_layout()
        savefig(joinpath(outdir, "basis_dist$(suffix).pdf"); dpi=300)
        close(fig)

        # =========================
        # FIGURE 3: RMSE
        # =========================
        fig = figure(figsize=(9,5))
        ax = gca()
        ax.set_xscale("log")
        ax.set_yscale("log")

        legend_handles_1d = Any[]
        legend_labels_1d  = String[]
        legend_handles_2d = Any[]
        legend_labels_2d  = String[]
        yvals_log = Float64[]

        for b in bases
            nm = basis_display_name(b)
            r1 = MULTI_BASIS_CACHE[(mode,1,b)]
            r2 = MULTI_BASIS_CACHE[(mode,2,b)]

            y1p = clamp.(get_rmse(r1), 1e-18, Inf)
            y2p = clamp.(get_rmse(r2), 1e-18, Inf)

            h1 = ax.scatter(r1.x, y1p; marker=markers[b], c="red", s=18, zorder=3)
            h2 = ax.scatter(r2.x, y2p; marker=markers[b], c="blue", s=18, zorder=3)

            push!(legend_handles_1d, h1)
            push!(legend_labels_1d, nm)
            push!(legend_handles_2d, h2)
            push!(legend_labels_2d, nm)

            append!(yvals_log, y1p)
            append!(yvals_log, y2p)
        end

        if mode == :rel
            ax.axvline(1.0, lw=2.0, ls=":", color="black", alpha=0.9)
        end

        set_log_ylim_reasonable!(ax, yvals_log; floor=1e-12)
        ax.grid(true, which="both", alpha=0.25)

        style_2d_axes!(ax; xlabel=xlab, ylabel=L"\mathrm{RMSE}")
        if mode == :rel
            split_dim_legend!(ax, legend_handles_1d, legend_labels_1d, legend_handles_2d, legend_labels_2d;
                              loc1="lower right", loc2="lower right",
                              anchor1=(0.66, 0.02), anchor2=(0.83, 0.02))
        else
            split_dim_legend!(ax, legend_handles_1d, legend_labels_1d, legend_handles_2d, legend_labels_2d;
                              loc1="upper left", loc2="upper left",
                              anchor1=(0.02, 0.98), anchor2=(0.15, 0.98))
        end

        tight_layout()
        savefig(joinpath(outdir, "basis_rmse$(suffix).pdf"); dpi=300)
        close(fig)
    end

    make_3_plots(:rel)
    make_3_plots(:abs)

    println("Saved 6 figures in: ", outdir)
    println("  ", joinpath(outdir, "basis_overlap_rel.pdf"))
    println("  ", joinpath(outdir, "basis_dist_rel.pdf"))
    println("  ", joinpath(outdir, "basis_rmse_rel.pdf"))
    println("  ", joinpath(outdir, "basis_overlap_abs.pdf"))
    println("  ", joinpath(outdir, "basis_dist_abs.pdf"))
    println("  ", joinpath(outdir, "basis_rmse_abs.pdf"))
end

# ----------------------------
# DEFAULT RUN:
# Fourier, Bessel, Legendre, Jacobi, Chebyshev I, Chebyshev II, Haar
# ----------------------------
bases_main = [
    :fourier,
    :bessel,
    :legendre,
    :jacobi,
    :chebyshev1,
    :chebyshev2,
    :haar
]

markers_main = Dict(
    :fourier     => "^",
    :bessel      => "o",
    :legendre    => "s",
    :jacobi      => "v",
    :chebyshev1  => "<",
    :chebyshev2  => ">",
    :haar        => "D",
)

plot_basis_comparison_all(bases_main, markers_main;
                          outdir="figs",
                          use_threads=true,
                          use_median=true)

# ----------------------------
# OPTIONAL RUN (COMMENTED OUT): Mathieu + PSWF only
# ----------------------------
# bases_special   = [:mathieu, :pswf]
# markers_special = Dict(:mathieu=>"*", :pswf=>"H")
# plot_basis_comparison_all(bases_special, markers_special;
#                           outdir="out_mathieu_pswf",
#                           use_threads=true,
#                           use_median=true)

σ-sweep (rel, 1D, Fourier) 100%|█████████████████████████| Time: 0:00:04

σ* (1D, Fourier     ) = 8.99628   ||ĉ0||=2.84487   p_eff=30



σ-sweep (rel, 2D, Fourier) 100%|█████████████████████████| Time: 0:01:02


σ* (2D, Fourier     ) = 0.781898   ||ĉ0||=0.289056   p_eff=164


σ-sweep (abs, 1D, Fourier) 100%|█████████████████████████| Time: 0:00:04
σ-sweep (abs, 2D, Fourier) 100%|█████████████████████████| Time: 0:00:59
σ-sweep (rel, 1D, Bessel) 100%|██████████████████████████| Time: 0:00:03


...

σ* (1D, Haar        ) = 9.11434   ||ĉ0||=2.83376   p_eff=29


σ-sweep (rel, 2D, Haar) 100%|████████████████████████████| Time: 0:01:17


σ* (2D, Haar        ) = 0.750478   ||ĉ0||=0.288227   p_eff=177


σ-sweep (abs, 1D, Haar) 100%|████████████████████████████| Time: 0:00:04
σ-sweep (abs, 2D, Haar) 100%|████████████████████████████| Time: 0:01:21


Saved 6 figures in: figs
  figs\basis_overlap_rel.pdf
  figs\basis_dist_rel.pdf
  figs\basis_rmse_rel.pdf
  figs\basis_overlap_abs.pdf
  figs\basis_dist_abs.pdf
  figs\basis_rmse_abs.pdf
